

## Objective
This notebook focuses on runtime artifact generation for serving in the recommendation system pipeline.

## Data Sources
Primary inputs referenced in this notebook include:
- main_data.csv
- item_catalog.csv
- item_catalog_needs_review.csv
- item_catalog_conflict_details.csv
- category_rule_artifacts.json
- category_allowed_mapping.csv
- category_rule_pairs.csv
- category_wise_catalog_items.csv

## Core Workflow
- Import dependencies and initialize configuration.
- Load source datasets into dataframes.
- Aggregate events/items to create behavioral features.
- Persist model/artifacts for reuse.
- Export processed outputs to CSV.
- Serialize python objects for runtime loading.

## Expected Outputs
- Processed CSV outputs for downstream stages.
- Serialized model/artifact files for runtime loading.

## Role in Pipeline
This notebook contributes notebook-specific assets/signals that feed later candidate generation, ranking, or retraining steps.



<!-- AUTO_CELL_EXPLANATION -->
### Cell 1: import json
**Explanation:** This cell executes logic related to `import json`.

**Possible Output:** Possible output: text/log/value print.


In [1]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

BASE_DIR = Path(r"C:\D drive\item_recommendation_model_create")
DATA_DIR = BASE_DIR / "data"

INPUT_FILE = DATA_DIR / "main_data.csv"

ITEM_CATALOG_OUTPUT_DIR = DATA_DIR / "item_catalog_output"
ITEM_CATALOG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ITEM_CATALOG_FILE = ITEM_CATALOG_OUTPUT_DIR / "item_catalog.csv"
ITEM_REVIEW_FILE = ITEM_CATALOG_OUTPUT_DIR / "item_catalog_needs_review.csv"
ITEM_CONFLICT_FILE = ITEM_CATALOG_OUTPUT_DIR / "item_catalog_conflict_details.csv"

CATEGORY_RULE_JSON_FILE = ITEM_CATALOG_OUTPUT_DIR / "category_rule_artifacts.json"
CATEGORY_ALLOWED_MAPPING_FILE = ITEM_CATALOG_OUTPUT_DIR / "category_allowed_mapping.csv"
CATEGORY_RULE_PAIRS_FILE = ITEM_CATALOG_OUTPUT_DIR / "category_rule_pairs.csv"

print("Input file:", INPUT_FILE)
print("Output folder:", ITEM_CATALOG_OUTPUT_DIR)
print("Input file exists:", INPUT_FILE.exists())

Input file: C:\D drive\item_recommendation_model_create\data\main_data.csv
Output folder: C:\D drive\item_recommendation_model_create\data\item_catalog_output
Input file exists: True


<!-- AUTO_CELL_EXPLANATION -->
### Cell 2: df = pd.read_csv(INPUT_FILE)
**Explanation:** This cell executes logic related to `df = pd.read_csv(INPUT_FILE)`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [2]:
df = pd.read_csv(INPUT_FILE)

df.columns = [c.strip() for c in df.columns]

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (1000000, 16)
Columns:
['transactionId', 'customerId', 'customerName', 'customerPersona', 'itemId', 'itemName', 'category', 'quantity', 'orderDate', 'isHoliday', 'isFestival', 'season', 'timeSlot', 'dayOfWeek', 'hour', 'month']


,transactionId,customerId,customerName,customerPersona,itemId,itemName,category,quantity,orderDate,isHoliday,isFestival,season,timeSlot,dayOfWeek,hour,month
0,1,23433,MD MOSSIN,family_grocery,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice,6,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
1,1,23433,MD MOSSIN,family_grocery,453,Saad Testy Bit Salt-100gm,pantry salt,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
2,1,23433,MD MOSSIN,family_grocery,15262,Cow Brain-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
3,1,23433,MD MOSSIN,family_grocery,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
4,1,23433,MD MOSSIN,family_grocery,13986,Cow Stomach-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1


<!-- AUTO_CELL_EXPLANATION -->
### Cell 3: required_cols = ["itemId", "itemName", "category"]
**Explanation:** This cell executes logic related to `required_cols = ["itemId", "itemName", "category"]`.

**Possible Output:** Possible output: text/log/value print.


In [3]:
required_cols = ["itemId", "itemName", "category"]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("Required columns found")

Required columns found


<!-- AUTO_CELL_EXPLANATION -->
### Cell 4: def normalize_text(value):
**Explanation:** This cell executes logic related to `def normalize_text(value):`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [4]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    
    return value


item_df = df.copy()

item_df["itemId"] = pd.to_numeric(item_df["itemId"], errors="coerce")
item_df["itemName"] = item_df["itemName"].apply(normalize_text)
item_df["category"] = item_df["category"].apply(normalize_text)

item_df = item_df.dropna(subset=["itemId"]).copy()
item_df = item_df[
    (item_df["itemName"] != "") &
    (item_df["category"] != "")
].copy()

item_df["itemId"] = item_df["itemId"].astype(int)

print("Clean item data shape:", item_df.shape)
display(item_df[["itemId", "itemName", "category"]].head())

Clean item data shape: (1000000, 16)


,itemId,itemName,category
0,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice
1,453,Saad Testy Bit Salt-100gm,pantry salt
2,15262,Cow Brain-K,Meat-Fresh
3,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils
4,13986,Cow Stomach-K,Meat-Fresh


<!-- AUTO_CELL_EXPLANATION -->
### Cell 5: item_name_counter = defaultdict(Counter)
**Explanation:** This cell executes logic related to `item_name_counter = defaultdict(Counter)`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [5]:
item_name_counter = defaultdict(Counter)
item_category_counter = defaultdict(Counter)
item_price_values = defaultdict(list)
item_quantity_values = defaultdict(list)

for _, row in item_df.iterrows():
    item_id = int(row["itemId"])
    item_name = row["itemName"]
    category = row["category"]
    
    item_name_counter[item_id][item_name] += 1
    item_category_counter[item_id][category] += 1
    
    if "price" in item_df.columns and pd.notna(row.get("price", np.nan)):
        item_price_values[item_id].append(float(row["price"]))
    
    if "quantity" in item_df.columns and pd.notna(row.get("quantity", np.nan)):
        item_quantity_values[item_id].append(float(row["quantity"]))

catalog_rows = []

for item_id in sorted(item_name_counter.keys()):
    canonical_name = item_name_counter[item_id].most_common(1)[0][0]
    canonical_category = item_category_counter[item_id].most_common(1)[0][0]
    
    name_variant_count = len(item_name_counter[item_id])
    category_variant_count = len(item_category_counter[item_id])
    total_rows_seen = sum(item_name_counter[item_id].values())
    
    avg_price = round(float(np.mean(item_price_values[item_id])), 2) if item_price_values[item_id] else np.nan
    min_price = round(float(np.min(item_price_values[item_id])), 2) if item_price_values[item_id] else np.nan
    max_price = round(float(np.max(item_price_values[item_id])), 2) if item_price_values[item_id] else np.nan
    avg_quantity = round(float(np.mean(item_quantity_values[item_id])), 2) if item_quantity_values[item_id] else np.nan
    
    catalog_rows.append({
        "itemId": item_id,
        "itemName": canonical_name,
        "category": canonical_category,
        "avgPrice": avg_price,
        "minPrice": min_price,
        "maxPrice": max_price,
        "avgQuantity": avg_quantity,
        "totalRowsSeen": total_rows_seen,
        "nameVariantCount": name_variant_count,
        "categoryVariantCount": category_variant_count,
        "reviewFlag": 1 if name_variant_count > 1 or category_variant_count > 1 else 0
    })

item_catalog_df = pd.DataFrame(catalog_rows)

print("Item catalog shape:", item_catalog_df.shape)
display(item_catalog_df.head(20))

Item catalog shape: (229, 11)


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,NaN,NaN,NaN,1.40,3052,1,1,0
1,27,Pran Toast-250g,Snacks-General,NaN,NaN,NaN,2.16,2122,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry,NaN,NaN,NaN,1.39,2085,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,NaN,NaN,NaN,1.12,2820,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,NaN,NaN,NaN,1.37,2816,1,1,0
5,133,Rin Advanced-500gm,Household-Laundry,NaN,NaN,NaN,1.41,2085,1,1,0
6,292,Super White Powder-1000g,Household-Laundry,NaN,NaN,NaN,1.41,2266,1,1,0
7,296,Chaka Advanced Ball Soap-125gm,Household-Laundry,NaN,NaN,NaN,1.39,2076,1,1,0
8,318,Musur Pulse Kangaroo -1Kg (sp),Pantry-Pulses,NaN,NaN,NaN,2.67,5192,1,1,0
9,332,ACI Pure Salt-1Kg,pantry salt,NaN,NaN,NaN,1.40,12698,1,1,0


<!-- AUTO_CELL_EXPLANATION -->
### Cell 6: needs_review_df = item_catalog_df[item_catalog_df["reviewFlag"] == ...
**Explanation:** This cell executes logic related to `needs_review_df = item_catalog_df[item_catalog_df["reviewFlag"] == ...`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [6]:
needs_review_df = item_catalog_df[item_catalog_df["reviewFlag"] == 1].copy()

conflict_rows = []

for item_id in sorted(item_name_counter.keys()):
    if len(item_name_counter[item_id]) > 1 or len(item_category_counter[item_id]) > 1:
        conflict_rows.append({
            "itemId": item_id,
            "itemNameVariants": json.dumps(dict(item_name_counter[item_id]), ensure_ascii=False),
            "categoryVariants": json.dumps(dict(item_category_counter[item_id]), ensure_ascii=False)
        })

conflict_details_df = pd.DataFrame(conflict_rows)

print("Needs review shape:", needs_review_df.shape)
print("Conflict details shape:", conflict_details_df.shape)

display(needs_review_df.head())
display(conflict_details_df.head())

Needs review shape: (0, 11)
Conflict details shape: (0, 0)


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag


""


<!-- AUTO_CELL_EXPLANATION -->
### Cell 7: all_valid_categories = [
**Explanation:** This cell executes logic related to `all_valid_categories = [`.

**Possible Output:** Possible output: text/log/value print.


In [7]:
all_valid_categories = [
    'Bakery-Bread', 'Beverage-Carbonated', 'Beverage-Hot', 'Beverage-Juice', 'Beverage-Water',
    'Chocolates-Sweets', 'Dairy-Milk', 'Dairy-Other', 'Desserts-Traditional', 'Dry-Fruits',
    'Fish-Fresh', 'Fruits-Fresh', 'Household-AirCare', 'Household-Cleaning', 'Household-Kitchen',
    'Household-Laundry', 'Household-Utility', 'Meat-Fresh', 'Meat-Processed', 'Pantry-Flour',
    'Pantry-Grains', 'Pantry-Oils', 'Pantry-Pulses', 'Pantry-Rice', 'Pantry-Sweeteners',
    'Personal-Care-Bath', 'Personal-Care-Cosmetics', 'Personal-Care-Hair', 'Personal-Care-Oral',
    'Personal-Care-Sanitary', 'Pickle', 'Protein-Egg', 'Snacks-General', 'Spices-Cooking',
    'Veg-Cooking', 'cleaning_clothes', 'clothing - male', 'clothing accessories female',
    'clothing accessories male', 'clothing babies', 'clothing female', 'general_cooking_vegetables',
    'household hygene', 'noodles_pasta_and_haleem', 'pantry salt', 'sauce item'
]

category_allowed_map = {}

category_allowed_map["Bakery-Bread"] = {
    "Morning": [
        "Beverage-Hot", "Dairy-Milk", "Dairy-Other", "Pantry-Flour", "Pantry-Oils",
        "Pantry-Sweeteners", "Protein-Egg", "general_cooking_vegetables"
    ],
    "Noon": [
        "Dairy-Other", "Dry-Fruits", "Meat-Processed", "Pantry-Sweeteners", "Protein-Egg",
        "Snacks-General", "Veg-Cooking", "general_cooking_vegetables", "pantry salt",
        "sauce item", "noodles_pasta_and_haleem"
    ],
    "Afternoon": [
        "Dairy-Other", "Dry-Fruits", "Meat-Processed", "Pantry-Sweeteners", "Protein-Egg",
        "Snacks-General", "Veg-Cooking", "general_cooking_vegetables", "pantry salt",
        "sauce item", "noodles_pasta_and_haleem"
    ],
    "Evening": [
        "Dairy-Other", "Dry-Fruits", "Meat-Processed", "Pantry-Sweeteners", "Protein-Egg",
        "Snacks-General", "Veg-Cooking", "general_cooking_vegetables", "pantry salt",
        "sauce item", "noodles_pasta_and_haleem"
    ],
    "Night": [
        "Dairy-Other", "Dry-Fruits", "Meat-Processed", "Pantry-Sweeteners", "Protein-Egg",
        "Snacks-General", "Veg-Cooking", "general_cooking_vegetables", "pantry salt",
        "sauce item", "noodles_pasta_and_haleem"
    ],
    "Any": [
        "Beverage-Hot", "Dairy-Milk", "Dairy-Other", "Pantry-Flour", "Pantry-Oils",
        "Pantry-Sweeteners", "Protein-Egg", "general_cooking_vegetables", "Dry-Fruits",
        "Meat-Processed", "Snacks-General", "Veg-Cooking", "pantry salt", "sauce item",
        "noodles_pasta_and_haleem"
    ]
}

category_allowed_map["Chocolates-Sweets"] = {
    "Any": all_valid_categories
}

category_allowed_map["Desserts-Traditional"] = {
    "Any": [
        "Dairy-Milk", "Pantry-Sweeteners", "Fruits-Fresh", "Dry-Fruits",
        "Pantry-Grains", "Protein-Egg", "pantry salt"
    ]
}

category_allowed_map["Fish-Fresh"] = {
    "Any": [
        "pantry salt", "Pantry-Oils", "general_cooking_vegetables",
        "Pantry-Rice", "Spices-Cooking", "Veg-Cooking"
    ],
    "strictDifferentCategoryOnly": True
}

household_group = [
    "Household-AirCare", "Household-Cleaning", "Household-Kitchen",
    "Household-Laundry", "Household-Utility", "household hygene"
]

for cat in household_group:
    category_allowed_map[cat] = {
        "Any": household_group
    }

category_allowed_map["Meat-Fresh"] = {
    "Any": [
        "pantry salt", "Pantry-Oils", "general_cooking_vegetables",
        "Pantry-Rice",
        "Spices-Cooking", "Beverage-Carbonated", "sauce item"
    ]
}

pantry_personal_household_group = [
    "Pantry-Flour", "Pantry-Grains", "Pantry-Oils", "Pantry-Pulses", "Pantry-Rice",
    "Pantry-Sweeteners", "pantry salt", "Personal-Care-Bath", "Personal-Care-Cosmetics",
    "Personal-Care-Hair", "Personal-Care-Oral", "Personal-Care-Sanitary", "household hygene"
]

for cat in pantry_personal_household_group:
    category_allowed_map[cat] = {
        "Any": pantry_personal_household_group
    }

category_allowed_map["Snacks-General"] = {
    "Any": [
        "Desserts-Traditional", "Dry-Fruits", "Dairy-Milk"
    ]
}

personal_care_group = [
    "Personal-Care-Bath", "Personal-Care-Cosmetics", "Personal-Care-Hair",
    "Personal-Care-Oral", "Personal-Care-Sanitary"
]

for cat in personal_care_group:
    category_allowed_map[cat] = {
        "Any": personal_care_group
    }

category_allowed_map["cleaning_clothes"] = {
    "Any": ["cleaning_clothes"]
}

category_allowed_map["clothing - male"] = {
    "Any": ["clothing accessories male", "clothing - male"],
    "priority": ["clothing accessories male"]
}

category_allowed_map["clothing accessories male"] = {
    "Any": ["clothing - male", "clothing accessories male"],
    "priority": ["clothing - male"]
}

category_allowed_map["clothing female"] = {
    "Any": ["clothing accessories female", "clothing female"],
    "priority": ["clothing accessories female"]
}

category_allowed_map["clothing accessories female"] = {
    "Any": ["clothing female", "clothing accessories female"],
    "priority": ["clothing female"]
}

category_allowed_map["clothing babies"] = {
    "Any": ["clothing babies"]
}

category_allowed_map["Meat-Processed"] = {
    "Any": ["sauce item", "noodles_pasta_and_haleem", "Beverage-Carbonated", "Snacks-General"]
}

category_allowed_map["Dairy-Milk"] = {
    "Any": ["Bakery-Bread", "Desserts-Traditional", "Snacks-General", "Beverage-Hot"]
}

category_allowed_map["Dairy-Other"] = {
    "Any": ["Bakery-Bread", "Desserts-Traditional", "Pantry-Sweeteners", "Protein-Egg"]
}

category_allowed_map["Dry-Fruits"] = {
    "Any": ["Desserts-Traditional", "Snacks-General", "Dairy-Milk"]
}

category_allowed_map["Fruits-Fresh"] = {
    "Any": ["Desserts-Traditional", "Dairy-Milk", "Beverage-Juice"]
}

category_allowed_map["Protein-Egg"] = {
    "Any": ["Bakery-Bread", "Pantry-Flour", "general_cooking_vegetables", "pantry salt"]
}

category_allowed_map["general_cooking_vegetables"] = {
    "Any": ["Pantry-Rice", "Pantry-Oils", "Spices-Cooking", "pantry salt", "Fish-Fresh", "Meat-Fresh", "Veg-Cooking"]
}

category_allowed_map["Veg-Cooking"] = {
    "Any": ["Pantry-Rice", "Pantry-Oils", "Spices-Cooking", "pantry salt", "Fish-Fresh", "general_cooking_vegetables"]
}

category_allowed_map["Spices-Cooking"] = {
    "Any": ["Pantry-Rice", "Pantry-Oils", "pantry salt", "Meat-Fresh", "Fish-Fresh", "general_cooking_vegetables"]
}

category_allowed_map["sauce item"] = {
    "Any": ["Meat-Fresh", "Meat-Processed", "Bakery-Bread", "noodles_pasta_and_haleem"]
}

category_allowed_map["noodles_pasta_and_haleem"] = {
    "Any": ["sauce item", "Snacks-General", "Beverage-Carbonated", "Bakery-Bread"]
}

category_allowed_map["Beverage-Carbonated"] = {
    "Any": ["Snacks-General", "Meat-Fresh", "Meat-Processed", "Bakery-Bread"]
}

category_allowed_map["Beverage-Hot"] = {
    "Any": ["Bakery-Bread", "Dairy-Milk", "Dairy-Other", "Pantry-Sweeteners"]
}

category_allowed_map["Beverage-Juice"] = {
    "Any": ["Snacks-General", "Fruits-Fresh", "Desserts-Traditional", "Bakery-Bread"]
}

category_allowed_map["Beverage-Water"] = {
    "Any": ["Snacks-General", "Bakery-Bread", "Pantry-Rice"]
}

category_allowed_map["Pickle"] = {
    "Any": ["Pantry-Rice", "Pantry-Oils", "Spices-Cooking", "general_cooking_vegetables"]
}

print("Total valid categories:", len(all_valid_categories))
print("Total rule categories:", len(category_allowed_map))

Total valid categories: 46
Total rule categories: 46


<!-- AUTO_CELL_EXPLANATION -->
### Cell 8: category_family_map = {
**Explanation:** This cell executes logic related to `category_family_map = {`.

**Possible Output:** Possible output: text/log/value print.


In [8]:
category_family_map = {
    "Bakery-Bread": "breakfast",
    "Beverage-Hot": "breakfast",
    "Dairy-Milk": "breakfast",
    "Dairy-Other": "breakfast",
    "Protein-Egg": "breakfast",

    "Meat-Fresh": "cooking",
    "Meat-Processed": "processed_food",
    "Fish-Fresh": "cooking",
    "Pantry-Rice": "cooking",
    "Pantry-Oils": "cooking",
    "Pantry-Pulses": "cooking",
    "Pantry-Flour": "cooking",
    "Pantry-Grains": "cooking",
    "Pantry-Sweeteners": "dessert",
    "pantry salt": "cooking",
    "Spices-Cooking": "cooking",
    "sauce item": "cooking",
    "Veg-Cooking": "cooking",
    "general_cooking_vegetables": "cooking",
    "Pickle": "cooking",

    "Snacks-General": "snack",
    "Chocolates-Sweets": "snack",
    "Beverage-Carbonated": "snack",
    "Beverage-Juice": "snack",
    "Beverage-Water": "beverage",
    "noodles_pasta_and_haleem": "snack",

    "Desserts-Traditional": "dessert",
    "Dry-Fruits": "dessert",
    "Fruits-Fresh": "dessert",

    "Household-AirCare": "household",
    "Household-Cleaning": "household",
    "Household-Kitchen": "household",
    "Household-Laundry": "household",
    "Household-Utility": "household",
    "household hygene": "household",

    "Personal-Care-Bath": "personal_care",
    "Personal-Care-Cosmetics": "personal_care",
    "Personal-Care-Hair": "personal_care",
    "Personal-Care-Oral": "personal_care",
    "Personal-Care-Sanitary": "personal_care",

    "cleaning_clothes": "clothing",
    "clothing - male": "clothing",
    "clothing accessories female": "clothing",
    "clothing accessories male": "clothing",
    "clothing babies": "clothing",
    "clothing female": "clothing"
}

print("Category family map created:", len(category_family_map))

Category family map created: 46


<!-- AUTO_CELL_EXPLANATION -->
### Cell 9: missing_rule_categories = sorted(set(all_valid_categories) - set(ca...
**Explanation:** This cell executes logic related to `missing_rule_categories = sorted(set(all_valid_categories) - set(ca...`.

**Possible Output:** Possible output: text/log/value print.


In [9]:
missing_rule_categories = sorted(set(all_valid_categories) - set(category_allowed_map.keys()))

if missing_rule_categories:
    print("Categories without rule:")
    print(missing_rule_categories)
else:
    print("Every category has a rule")

Every category has a rule


<!-- AUTO_CELL_EXPLANATION -->
### Cell 10: rule_rows = []
**Explanation:** This cell executes logic related to `rule_rows = []`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [10]:
rule_rows = []

for source_category, rule_info in category_allowed_map.items():
    for rule_key, allowed_categories in rule_info.items():
        if rule_key in ["strictDifferentCategoryOnly", "priority"]:
            continue
        
        for rank, target_category in enumerate(allowed_categories, start=1):
            rule_rows.append({
                "sourceCategory": source_category,
                "timeSlot": rule_key,
                "allowedTargetCategory": target_category,
                "rank": rank,
                "strictDifferentCategoryOnly": bool(rule_info.get("strictDifferentCategoryOnly", False)),
                "priorityCategories": json.dumps(rule_info.get("priority", []), ensure_ascii=False)
            })

category_rule_pairs_df = pd.DataFrame(rule_rows)

category_allowed_mapping_df = (
    category_rule_pairs_df
    .groupby(["sourceCategory", "timeSlot"], as_index=False)
    .agg(
        allowedTargetCategories=("allowedTargetCategory", lambda x: json.dumps(list(x), ensure_ascii=False)),
        strictDifferentCategoryOnly=("strictDifferentCategoryOnly", "max"),
        priorityCategories=("priorityCategories", "first")
    )
)

display(category_rule_pairs_df.head(30))
display(category_allowed_mapping_df.head(30))

,sourceCategory,timeSlot,allowedTargetCategory,rank,strictDifferentCategoryOnly,priorityCategories
0,Bakery-Bread,Morning,Beverage-Hot,1,False,[]
1,Bakery-Bread,Morning,Dairy-Milk,2,False,[]
2,Bakery-Bread,Morning,Dairy-Other,3,False,[]
3,Bakery-Bread,Morning,Pantry-Flour,4,False,[]
4,Bakery-Bread,Morning,Pantry-Oils,5,False,[]
5,Bakery-Bread,Morning,Pantry-Sweeteners,6,False,[]
6,Bakery-Bread,Morning,Protein-Egg,7,False,[]
7,Bakery-Bread,Morning,general_cooking_vegetables,8,False,[]
8,Bakery-Bread,Noon,Dairy-Other,1,False,[]
9,Bakery-Bread,Noon,Dry-Fruits,2,False,[]


,sourceCategory,timeSlot,allowedTargetCategories,strictDifferentCategoryOnly,priorityCategories
0,Bakery-Bread,Afternoon,"[""Dairy-Other"", ""Dry-Fruits"", ""Meat-Processed"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""Snacks-General"", ""Veg-Cooking"", ""general_cooking_vegetables"",...",False,[]
1,Bakery-Bread,Any,"[""Beverage-Hot"", ""Dairy-Milk"", ""Dairy-Other"", ""Pantry-Flour"", ""Pantry-Oils"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""general_cooking_vegetables"", ""Dr...",False,[]
2,Bakery-Bread,Evening,"[""Dairy-Other"", ""Dry-Fruits"", ""Meat-Processed"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""Snacks-General"", ""Veg-Cooking"", ""general_cooking_vegetables"",...",False,[]
3,Bakery-Bread,Morning,"[""Beverage-Hot"", ""Dairy-Milk"", ""Dairy-Other"", ""Pantry-Flour"", ""Pantry-Oils"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""general_cooking_vegetables""]",False,[]
4,Bakery-Bread,Night,"[""Dairy-Other"", ""Dry-Fruits"", ""Meat-Processed"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""Snacks-General"", ""Veg-Cooking"", ""general_cooking_vegetables"",...",False,[]
5,Bakery-Bread,Noon,"[""Dairy-Other"", ""Dry-Fruits"", ""Meat-Processed"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""Snacks-General"", ""Veg-Cooking"", ""general_cooking_vegetables"",...",False,[]
6,Beverage-Carbonated,Any,"[""Snacks-General"", ""Meat-Fresh"", ""Meat-Processed"", ""Bakery-Bread""]",False,[]
7,Beverage-Hot,Any,"[""Bakery-Bread"", ""Dairy-Milk"", ""Dairy-Other"", ""Pantry-Sweeteners""]",False,[]
8,Beverage-Juice,Any,"[""Snacks-General"", ""Fruits-Fresh"", ""Desserts-Traditional"", ""Bakery-Bread""]",False,[]
9,Beverage-Water,Any,"[""Snacks-General"", ""Bakery-Bread"", ""Pantry-Rice""]",False,[]


<!-- AUTO_CELL_EXPLANATION -->
### Cell 11: def get_allowed_categories(source_category, time_slot=None):
**Explanation:** This cell executes logic related to `def get_allowed_categories(source_category, time_slot=None):`.

**Possible Output:** Possible output: text/log/value print.


In [11]:
def get_allowed_categories(source_category, time_slot=None):
    if source_category not in category_allowed_map:
        return []
    
    rule = category_allowed_map[source_category]
    
    if time_slot is not None and time_slot in rule:
        return rule[time_slot]
    
    return rule.get("Any", [])


def filter_candidates_by_category_rules(
    input_categories,
    candidate_df,
    candidate_category_col="category",
    candidate_item_id_col="itemId",
    time_slot=None,
    max_per_category=1
):
    if isinstance(input_categories, str):
        input_categories = [input_categories]
    
    allowed_categories = []
    strict_different_category_only = False
    priority_categories = []
    
    for source_category in input_categories:
        allowed_categories.extend(get_allowed_categories(source_category, time_slot=time_slot))
        
        rule = category_allowed_map.get(source_category, {})
        
        if rule.get("strictDifferentCategoryOnly", False):
            strict_different_category_only = True
        
        priority_categories.extend(rule.get("priority", []))
    
    allowed_categories = list(dict.fromkeys(allowed_categories))
    priority_categories = list(dict.fromkeys(priority_categories))
    
    filtered = candidate_df[
        candidate_df[candidate_category_col].isin(allowed_categories)
    ].copy()
    
    if strict_different_category_only:
        filtered = filtered[
            ~filtered[candidate_category_col].isin(input_categories)
        ].copy()
        
        filtered = (
            filtered
            .sort_values(candidate_item_id_col)
            .groupby(candidate_category_col, as_index=False)
            .head(max_per_category)
        )
    
    if priority_categories:
        priority_map = {cat: rank for rank, cat in enumerate(priority_categories)}
        filtered["rulePriority"] = filtered[candidate_category_col].map(priority_map).fillna(999).astype(int)
        filtered = filtered.sort_values(["rulePriority", candidate_category_col, candidate_item_id_col])
    else:
        filtered["rulePriority"] = 999
    
    return filtered.reset_index(drop=True)


print("Strict recommendation filter function ready")

Strict recommendation filter function ready


<!-- AUTO_CELL_EXPLANATION -->
### Cell 12: test_candidate_df = item_catalog_df[["itemId", "itemName", "categor...
**Explanation:** This cell executes logic related to `test_candidate_df = item_catalog_df[["itemId", "itemName", "categor...`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [12]:
test_candidate_df = item_catalog_df[["itemId", "itemName", "category"]].copy()

test_cases = [
    ("Fish-Fresh", None),
    ("Meat-Fresh", None),
    ("Household-Kitchen", None),
    ("Personal-Care-Bath", None),
    ("clothing - male", None),
    ("Bakery-Bread", "Morning"),
    ("Bakery-Bread", "Evening"),
    ("Snacks-General", None)
]

for source_category, time_slot in test_cases:
    print("\nSource category:", source_category)
    print("Time slot:", time_slot)
    print("Allowed categories:", get_allowed_categories(source_category, time_slot))
    
    test_output = filter_candidates_by_category_rules(
        input_categories=[source_category],
        candidate_df=test_candidate_df,
        time_slot=time_slot
    )
    
    display(test_output.head(10))


Source category: Fish-Fresh
Time slot: None
Allowed categories: ['pantry salt', 'Pantry-Oils', 'general_cooking_vegetables', 'Pantry-Rice', 'Spices-Cooking', 'Veg-Cooking']


,itemId,itemName,category,rulePriority
0,332,ACI Pure Salt-1Kg,pantry salt,999
1,551,Cumin Powder-100gm,Spices-Cooking,999
2,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice,999
3,2038,Teer Soya Oil 2Lt(Bottle),Pantry-Oils,999
4,6848,Garlic China Big-(kg),general_cooking_vegetables,999
5,14815,Tomato L.C(kg),Veg-Cooking,999



Source category: Meat-Fresh
Time slot: None
Allowed categories: ['pantry salt', 'Pantry-Oils', 'general_cooking_vegetables', 'Pantry-Rice', 'Spices-Cooking', 'Beverage-Carbonated', 'sauce item']


,itemId,itemName,category,rulePriority
0,332,ACI Pure Salt-1Kg,pantry salt,999
1,453,Saad Testy Bit Salt-100gm,pantry salt,999
2,551,Cumin Powder-100gm,Spices-Cooking,999
3,604,Cassia Leaf-100gm (Tespata),Spices-Cooking,999
4,704,Radhuni Spices Cumin-200gm,Spices-Cooking,999
5,708,Radhuni Chatpati Masala 50Gm,Spices-Cooking,999
6,782,Panch Foron-100g,Spices-Cooking,999
7,878,Radhuni Spices (Corinder)-100gm,Spices-Cooking,999
8,886,Radhuni Chilli Powder -200gm,Spices-Cooking,999
9,890,Radhuni Garam Masala-40gm,Spices-Cooking,999



Source category: Household-Kitchen
Time slot: None
Allowed categories: ['Household-AirCare', 'Household-Cleaning', 'Household-Kitchen', 'Household-Laundry', 'Household-Utility', 'household hygene']


...

,itemId,itemName,category,rulePriority
0,2038,Teer Soya Oil 2Lt(Bottle),Pantry-Oils,999
1,2043,Rupchanda Soyabean-5Ltr,Pantry-Oils,999
2,3182,Starship Chocolate Milk-200ml,Dairy-Milk,999
3,3185,Marks Milk Shake AS-125,Dairy-Milk,999
4,3787,Rose Water-180ml,Dairy-Other,999
5,3809,Arla Dano Daily Pushti-500gm,Dairy-Milk,999
6,3831,Milk Vita Liquid Milk-1Ltr,Dairy-Milk,999
7,3858,Pran Sweetened Yogurt-100gm,Dairy-Other,999
8,3952,Pran UHT Milk-200ml,Dairy-Milk,999
9,4958,Teer Sugar-1Kg,Pantry-Sweeteners,999



Source category: Bakery-Bread
Time slot: Evening
Allowed categories: ['Dairy-Other', 'Dry-Fruits', 'Meat-Processed', 'Pantry-Sweeteners', 'Protein-Egg', 'Snacks-General', 'Veg-Cooking', 'general_cooking_vegetables', 'pantry salt', 'sauce item', 'noodles_pasta_and_haleem']


,itemId,itemName,category,rulePriority
0,27,Pran Toast-250g,Snacks-General,999
1,332,ACI Pure Salt-1Kg,pantry salt,999
2,445,Savory Haleem Mix-200gm,noodles_pasta_and_haleem,999
3,453,Saad Testy Bit Salt-100gm,pantry salt,999
4,545,Golden Raisin (Kismis)-200gm,Dry-Fruits,999
5,762,Maggi Saad -E -Magic S Mix-48g,noodles_pasta_and_haleem,999
6,864,Pran Haleem Mix(200g),noodles_pasta_and_haleem,999
7,3612,Italiano Sweet Chilli Sauce-340g,sauce item,999
8,3739,Hot Pran Tomato Sauce-500gm,sauce item,999
9,3787,Rose Water-180ml,Dairy-Other,999



Source category: Snacks-General
Time slot: None
Allowed categories: ['Desserts-Traditional', 'Dry-Fruits', 'Dairy-Milk']


,itemId,itemName,category,rulePriority
0,545,Golden Raisin (Kismis)-200gm,Dry-Fruits,999
1,3182,Starship Chocolate Milk-200ml,Dairy-Milk,999
2,3185,Marks Milk Shake AS-125,Dairy-Milk,999
3,3809,Arla Dano Daily Pushti-500gm,Dairy-Milk,999
4,3831,Milk Vita Liquid Milk-1Ltr,Dairy-Milk,999
5,3952,Pran UHT Milk-200ml,Dairy-Milk,999
6,12637,Ramisa-Nakshi Pitha Box-4 Pcs,Desserts-Traditional,999
7,12674,Ramisa Sweet Munakka-200gm,Dry-Fruits,999
8,12963,White Kazu Badam-50g,Dry-Fruits,999
9,16230,Polar Ice-Cream Kheer-1Ltr,Desserts-Traditional,999


<!-- AUTO_CELL_EXPLANATION -->
### Cell 13: item_catalog_df.to_csv(ITEM_CATALOG_FILE, index=False)
**Explanation:** This cell executes logic related to `item_catalog_df.to_csv(ITEM_CATALOG_FILE, index=False)`.

**Possible Output:** Possible output: text/log/value print.


In [14]:
item_catalog_df.to_csv(ITEM_CATALOG_FILE, index=False)
needs_review_df.to_csv(ITEM_REVIEW_FILE, index=False)
conflict_details_df.to_csv(ITEM_CONFLICT_FILE, index=False)

category_rule_pairs_df.to_csv(CATEGORY_RULE_PAIRS_FILE, index=False)
category_allowed_mapping_df.to_csv(CATEGORY_ALLOWED_MAPPING_FILE, index=False)

rule_artifacts = {
    "all_valid_categories": all_valid_categories,
    "category_allowed_map": category_allowed_map,
    "category_family_map": category_family_map,
    "notes": {
        "Fish-Fresh": "Only allowed cooking related categories, different target category enforced",
        "Meat-Fresh": "Only allowed cooking and beverage categories",
        "Household group": "Household categories recommend only within household group",
        "Personal care group": "Personal care categories recommend only within personal care group",
        "Clothing group": "Gender and baby category matching preserved",
        "Chocolates-Sweets": "Open category because user said no fixed relevancy",
        "Dynamic mixed basket": "Category family map is used for balanced recommendation across mixed baskets"
    }
}

with open(CATEGORY_RULE_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(rule_artifacts, f, indent=4, ensure_ascii=False)

print("Saved files:")
print(ITEM_CATALOG_FILE)
print(ITEM_REVIEW_FILE)
print(ITEM_CONFLICT_FILE)
print(CATEGORY_RULE_JSON_FILE)
print(CATEGORY_ALLOWED_MAPPING_FILE)
print(CATEGORY_RULE_PAIRS_FILE)

Saved files:
C:\D drive\item_recommendation_model_create\data\item_catalog_output\item_catalog.csv
C:\D drive\item_recommendation_model_create\data\item_catalog_output\item_catalog_needs_review.csv
C:\D drive\item_recommendation_model_create\data\item_catalog_output\item_catalog_conflict_details.csv
C:\D drive\item_recommendation_model_create\data\item_catalog_output\category_rule_artifacts.json
C:\D drive\item_recommendation_model_create\data\item_catalog_output\category_allowed_mapping.csv
C:\D drive\item_recommendation_model_create\data\item_catalog_output\category_rule_pairs.csv


<!-- AUTO_CELL_EXPLANATION -->
### Cell 14: catalog_check = pd.read_csv(ITEM_CATALOG_FILE)
**Explanation:** This cell executes logic related to `catalog_check = pd.read_csv(ITEM_CATALOG_FILE)`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [15]:
catalog_check = pd.read_csv(ITEM_CATALOG_FILE)
rule_pairs_check = pd.read_csv(CATEGORY_RULE_PAIRS_FILE)
allowed_mapping_check = pd.read_csv(CATEGORY_ALLOWED_MAPPING_FILE)

print("Catalog shape:", catalog_check.shape)
print("Rule pairs shape:", rule_pairs_check.shape)
print("Allowed mapping shape:", allowed_mapping_check.shape)

display(catalog_check.head())
display(rule_pairs_check.head())
display(allowed_mapping_check.head())

Catalog shape: (229, 11)
Rule pairs shape: (373, 6)
Allowed mapping shape: (51, 5)


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,NaN,NaN,NaN,1.40,3052,1,1,0
1,27,Pran Toast-250g,Snacks-General,NaN,NaN,NaN,2.16,2122,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry,NaN,NaN,NaN,1.39,2085,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,NaN,NaN,NaN,1.12,2820,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,NaN,NaN,NaN,1.37,2816,1,1,0


,sourceCategory,timeSlot,allowedTargetCategory,rank,strictDifferentCategoryOnly,priorityCategories
0,Bakery-Bread,Morning,Beverage-Hot,1,False,[]
1,Bakery-Bread,Morning,Dairy-Milk,2,False,[]
2,Bakery-Bread,Morning,Dairy-Other,3,False,[]
3,Bakery-Bread,Morning,Pantry-Flour,4,False,[]
4,Bakery-Bread,Morning,Pantry-Oils,5,False,[]


,sourceCategory,timeSlot,allowedTargetCategories,strictDifferentCategoryOnly,priorityCategories
0,Bakery-Bread,Afternoon,"[""Dairy-Other"", ""Dry-Fruits"", ""Meat-Processed"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""Snacks-General"", ""Veg-Cooking"", ""general_cooking_vegetables"",...",False,[]
1,Bakery-Bread,Any,"[""Beverage-Hot"", ""Dairy-Milk"", ""Dairy-Other"", ""Pantry-Flour"", ""Pantry-Oils"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""general_cooking_vegetables"", ""Dr...",False,[]
2,Bakery-Bread,Evening,"[""Dairy-Other"", ""Dry-Fruits"", ""Meat-Processed"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""Snacks-General"", ""Veg-Cooking"", ""general_cooking_vegetables"",...",False,[]
3,Bakery-Bread,Morning,"[""Beverage-Hot"", ""Dairy-Milk"", ""Dairy-Other"", ""Pantry-Flour"", ""Pantry-Oils"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""general_cooking_vegetables""]",False,[]
4,Bakery-Bread,Night,"[""Dairy-Other"", ""Dry-Fruits"", ""Meat-Processed"", ""Pantry-Sweeteners"", ""Protein-Egg"", ""Snacks-General"", ""Veg-Cooking"", ""general_cooking_vegetables"",...",False,[]


<!-- AUTO_CELL_EXPLANATION -->
### Cell 15: category_wise_catalog_items = (
**Explanation:** This cell executes logic related to `category_wise_catalog_items = (`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [16]:
category_wise_catalog_items = (
    item_catalog_df
    .groupby("category")
    .agg(
        totalItems=("itemId", "nunique"),
        itemIds=("itemId", lambda x: json.dumps(list(x), ensure_ascii=False)),
        itemNames=("itemName", lambda x: json.dumps(list(x), ensure_ascii=False))
    )
    .reset_index()
    .sort_values("totalItems", ascending=False)
)

CATEGORY_WISE_ITEM_FILE = ITEM_CATALOG_OUTPUT_DIR / "category_wise_catalog_items.csv"

category_wise_catalog_items.to_csv(CATEGORY_WISE_ITEM_FILE, index=False)

print("Saved:", CATEGORY_WISE_ITEM_FILE)
display(category_wise_catalog_items)

Saved: C:\D drive\item_recommendation_model_create\data\item_catalog_output\category_wise_catalog_items.csv


,category,totalItems,itemIds,itemNames
32,Snacks-General,19,"[27, 4840, 4842, 4845, 4846, 4857, 4965, 7510, 7514, 7516, 12625, 12627, 12629, 12664, 12778, 13620, 13846, 13863, 30982]","[""Pran Toast-250g"", ""Sun Chips Mix Masala-35gm"", ""Sun Chips Wasabi-35g"", ""Sun Chips Tomato Tango 38g"", ""Sun Chips Garlic & Chili 38g"", ""Kurkure Ch..."
34,Veg-Cooking,13,"[14815, 14821, 14843, 14964, 14968, 14969, 14975, 14983, 14995, 14997, 14998, 15104, 15279]","[""Tomato L.C(kg)"", ""Green Pumpkin-pc"", ""Cucumber Deshi (kg)"", ""Green Papaya(kg)"", ""Eye Of Arum(Tall) (Kachurmukhi)-kg"", ""Carrot (Deshi) kg"", ""Caps..."
42,household hygene,12,"[24707, 24715, 24717, 25519, 25524, 25567, 25569, 25571, 32467, 32468, 42620, 42915]","[""Bashu. Toilet Tissue (White)"", ""Bashun Paper Towel 1Ply- 250Pcs"", ""B Toilet Tissue Extra Saving Pack 4Rolls"", ""Dettol Skincare HW-170ml(Refil)"",..."
43,noodles_pasta_and_haleem,11,"[445, 762, 864, 7202, 26536, 26607, 26615, 26627, 26628, 26634, 26766]","[""Savory Haleem Mix-200gm"", ""Maggi Saad -E -Magic S Mix-48g"", ""Pran Haleem Mix(200g)"", ""EM Pasta Vite-400gm"", ""Knorr Soup-24gm"", ""Doodles Stick No..."
7,Dairy-Other,11,"[3787, 3858, 6838, 7017, 7055, 7075, 7182, 7196, 15275, 17809, 25437]","[""Rose Water-180ml"", ""Pran Sweetened Yogurt-100gm"", ""Aarong Probiotic Yogurt-500gm"", ""Good Life Mozzarella Cheese-200gm"", ""Lactima Cbeddar Slice C..."
33,Spices-Cooking,11,"[551, 604, 704, 708, 782, 878, 886, 890, 891, 12823, 12834]","[""Cumin Powder-100gm"", ""Cassia Leaf-100gm (Tespata)"", ""Radhuni Spices Cumin-200gm"", ""Radhuni Chatpati Masala 50Gm"", ""Panch Foron-100g"", ""Radhuni S..."
10,Fish-Fresh,8,"[13901, 13917, 13921, 13922, 13930, 13958, 14045, 14046]","[""Katol Fish (Small)-(MNK)"", ""Rui Fish (XL)-(HB)"", ""Rui Fish-M-(MNK)"", ""Rui Fish (S1)-(MNK)"", ""Poa Fish Small-(MNK)"", ""Shrimp Fish Bagda"", ""Pabda ..."
11,Fruits-Fresh,8,"[12587, 14844, 14984, 15131, 15263, 15265, 15266, 15296]","[""Calender Pineapple"", ""Banana Green Small-kg"", ""Lemon Elachi-Pc"", ""Atta Fruits"", ""Malta (Yellow)-kg"", ""Naspati(kg)"", ""Apple (Fuji)"", ""Apple(Gala)""]"
1,Beverage-Carbonated,8,"[22234, 22235, 22237, 22241, 22251, 22258, 22259, 22260]","[""Sprite-1.25 Lt(Pet)"", ""Coca Cola-2.25Lt(Pet)"", ""Coca Cola-600ml(Pet)"", ""Kinley Club Soda -600ml"", ""Mountan Dew-250ml Can"", ""Pepsi 600ml"", ""Mirin..."
28,Personal-Care-Oral,7,"[2143, 2149, 2220, 2255, 4709, 4767, 4889]","[""Sensodyne Toothbrush Soft"", ""Sensodyne Deep Clean Toothbrush"", ""Ambition Tooth brush"", ""Pepsodent Gremichek+ Toothpaste-100gm"", ""Dabur Meswak Pa..."
